In [ ]:
import pandas as pd
import numpy as np

data_dir = '../data/'

# ==========================================
# BƯỚC 1: ĐỌC VÀ LÀM SẠCH 13 BẢNG
# ==========================================
df_products    = pd.read_csv(f'{data_dir}products.csv')
df_customers   = pd.read_csv(f'{data_dir}customers.csv')
df_promotions  = pd.read_csv(f'{data_dir}promotions.csv')
df_geography   = pd.read_csv(f'{data_dir}geography.csv')
df_orders      = pd.read_csv(f'{data_dir}orders.csv')
df_order_items = pd.read_csv(f'{data_dir}order_items.csv', dtype={'promo_id_2': str}, low_memory=False)
df_payments    = pd.read_csv(f'{data_dir}payments.csv')
df_shipments   = pd.read_csv(f'{data_dir}shipments.csv')
df_returns     = pd.read_csv(f'{data_dir}returns.csv')
df_reviews     = pd.read_csv(f'{data_dir}reviews.csv')
df_inventory   = pd.read_csv(f'{data_dir}inventory.csv')
df_web_traffic = pd.read_csv(f'{data_dir}web_traffic.csv')
# [FIX 1] Load thêm sales.csv cho Phần 3 - Forecasting
df_sales       = pd.read_csv(f'{data_dir}sales.csv')

# Làm sạch tên cột cho toàn bộ bảng
dataframes = [df_products, df_customers, df_promotions, df_geography,
              df_orders, df_order_items, df_payments, df_shipments,
              df_returns, df_reviews, df_inventory, df_web_traffic, df_sales]

for df in dataframes:
    df.columns = df.columns.str.strip().str.lower().str.replace('\ufeff', '')

print("✅ Đã load và làm sạch tên cột cho 13 bảng!")

# ==========================================
# BƯỚC 2: TẠO XƯƠNG SỐNG (LỚP TRANSACTION)
# ==========================================
df_master = pd.merge(df_order_items, df_orders, on='order_id', how='left')

# [FIX 2] payments có cột payment_method trùng với orders → dùng suffix để phân biệt
df_master = pd.merge(
    df_master,
    df_payments.rename(columns={'payment_method': 'payment_method_payments'}),
    on='order_id',
    how='left'
)

df_master = pd.merge(df_master, df_shipments, on='order_id', how='left')
df_master = pd.merge(df_master, df_returns, on=['order_id', 'product_id'], how='left')

# [FIX 3] reviews: giữ lại customer_id để có thể kiểm tra chéo nếu cần,
#          dùng suffix thay vì drop thẳng
df_master = pd.merge(
    df_master,
    df_reviews.rename(columns={'customer_id': 'reviewer_customer_id'}),
    on=['order_id', 'product_id'],
    how='left'
)

# ==========================================
# BƯỚC 3: ĐẮP THÔNG TIN (LỚP MASTER)
# ==========================================
df_master = pd.merge(df_master, df_products, on='product_id', how='left')
df_master = pd.merge(df_master, df_customers, on='customer_id', how='left', suffixes=('', '_cust'))

# Địa lý giao hàng (dùng zip từ orders)
df_master = pd.merge(
    df_master,
    df_geography.rename(columns={
        'zip':      'zip_delivery',
        'city':     'city_delivery',
        'region':   'region_delivery',
        'district': 'district_delivery'
    }),
    left_on='zip',
    right_on='zip_delivery',
    how='left'
)

# Địa lý khách hàng (dùng zip từ customers)
cust_zip_col = 'zip_cust' if 'zip_cust' in df_master.columns else 'zip'
df_master = pd.merge(
    df_master,
    df_geography.rename(columns={
        'zip':      'zip_customer',
        'city':     'city_customer',
        'region':   'region_customer',
        'district': 'district_customer'
    }),
    left_on=cust_zip_col,
    right_on='zip_customer',
    how='left'
)

# Khuyến mãi chính (promo_id)
df_master = pd.merge(
    df_master,
    df_promotions.add_prefix('promo1_').rename(columns={'promo1_promo_id': 'promo_id'}),
    on='promo_id',
    how='left'
)

# [FIX 4] Gộp thêm khuyến mãi thứ hai (promo_id_2) — đề có cột này trong order_items
df_master = pd.merge(
    df_master,
    df_promotions.add_prefix('promo2_').rename(columns={'promo2_promo_id': 'promo_id_2'}),
    on='promo_id_2',
    how='left'
)

# ==========================================
# BƯỚC 4: XỬ LÝ THỜI GIAN & NỐI LỚP VẬN HÀNH
# ==========================================
df_master['order_date']      = pd.to_datetime(df_master['order_date'], errors='coerce')
df_web_traffic['date']       = pd.to_datetime(df_web_traffic['date'],  errors='coerce')

# [FIX 5] Web Traffic: chỉ join theo ngày (date) — KHÔNG join theo traffic_source
#          vì web_traffic là daily aggregate, không có quan hệ 1:1 với order_source
df_web_agg = (
    df_web_traffic
    .groupby('date', as_index=False)
    .agg(
        sessions=('sessions', 'sum'),
        unique_visitors=('unique_visitors', 'sum'),
        page_views=('page_views', 'sum'),
        bounce_rate=('bounce_rate', 'mean'),
        avg_session_duration_sec=('avg_session_duration_sec', 'mean')
    )
)

df_master = pd.merge(
    df_master,
    df_web_agg,
    left_on='order_date',
    right_on='date',
    how='left'
).drop(columns=['date'])

# Inventory: join theo product_id + year + month
df_master['order_year']  = df_master['order_date'].dt.year
df_master['order_month'] = df_master['order_date'].dt.month

cols_to_drop_inv = ['product_name', 'category', 'segment', 'snapshot_date']
inventory_clean  = df_inventory.drop(
    columns=[c for c in cols_to_drop_inv if c in df_inventory.columns]
)

df_master = pd.merge(
    df_master,
    inventory_clean,
    left_on=['product_id', 'order_year', 'order_month'],
    right_on=['product_id', 'year', 'month'],
    how='left'
).drop(columns=['year', 'month'])

# ==========================================
# BƯỚC 5: CHUẨN BỊ SALES DATA (PHẦN 3 - FORECASTING)
# ==========================================
# [FIX 1 tiếp theo] Chuẩn hóa và xuất riêng sales train
df_sales['date'] = pd.to_datetime(df_sales['date'], errors='coerce')
df_sales = df_sales.sort_values('date').reset_index(drop=True)

print(f"📅 Sales train: {df_sales['date'].min().date()} → {df_sales['date'].max().date()}")
print(f"   Số dòng: {len(df_sales)}")

# ==========================================
# BƯỚC 6: XUẤT FILE
# ==========================================
print(f"\n🎉 THÀNH CÔNG! Master Data — Số dòng: {len(df_master)} | Số cột: {len(df_master.columns)}")

# Kiểm tra duplicate columns
dup_cols = df_master.columns[df_master.columns.duplicated()].tolist()
if dup_cols:
    print(f"⚠️  Cột bị trùng: {dup_cols}")
else:
    print("✅ Không có cột bị trùng tên.")

df_master.to_csv(f'{data_dir}final_master_data.csv', index=False)
df_sales.to_csv(f'{data_dir}sales_train_clean.csv',  index=False)

print(f"📦 Đã xuất: final_master_data.csv")
print(f"📦 Đã xuất: sales_train_clean.csv")